# NAMD-FEP — notebook 0 · Binder setup & environment check

**Run this notebook first.** It checks what the current kernel has, installs anything
missing, and confirms the `rbfe` / `rbfe-prep` commands work. Every other notebook in
this series assumes what is verified here.

This repository is designed to run on top of
[quantaosun/labodock_binder](https://github.com/quantaosun/labodock_binder), so the
starting point of the workflow is assumed to be a **docked pose or a crystal
structure** of a protein–ligand complex.

| notebook | step | needs |
|---|---|---|
| **00** (this one) | environment | nothing |
| 01 | visualise the complex | this one |
| 02 | prepare & sanitise inputs | 01 |
| 03 | hybrid ligand + system build | 02 |
| 04 | run / monitor production | 03 (+ a GPU for the run itself) |
| 05 | equilibration & free-energy analysis | 04, or the shipped 6I5I example |

> **Free Binder sessions have no GPU and a ~2 GB RAM limit.** Everything *except*
> the actual NAMD production run (notebook 04) works there: preparation,
> visualisation, sanity checks, and analysis of the shipped 6I5I example data.
> The production run itself must be launched on a GPU machine — notebook 04 shows
> how, and how to keep monitoring it.

## 0.1 What each piece is for

| package / tool | used by | install |
|---|---|---|
| `numpy` | the analysers (`rbfe.analysis`, `rbfe.estimators`) | conda/pip |
| `matplotlib` | the plots in notebooks 04–05 | conda/pip |
| `nglview` | the interactive 3D viewers | pip (nbextension enabled below) |
| `rdkit` | `rbfe similarity`, `rbfe-prep ligand --smiles`, 2D depictions | conda (`conda-forge::rdkit`) |
| `obabel` | ligand extraction / hydrogen completion in notebook 02 | conda (`conda-forge::openbabel`) |
| `acpype` | ligand parameterisation (AM1-BCC) in notebook 02 | pip (`acpype`) |
| NAMD 3.x + VMD | the build (notebook 03) and the run (notebook 04) | **not** installable here — point `$NAMD` / `$VMD` at a GPU box |

## 0.2 Check what is already installed

In [ ]:
import importlib, shutil, sys

print(f"python {sys.version.split()[0]}  ({sys.executable})")
print()
print(f"{'package':<12} {'status'}")
print('-' * 40)
for mod in ("numpy", "matplotlib", "nglview", "rdkit"):
    try:
        m = importlib.import_module(mod)
        print(f"{mod:<12} {getattr(m, '__version__', 'ok')}")
    except ImportError:
        print(f"{mod:<12} MISSING")
print()
print(f"{'tool':<12} {'status'}")
print('-' * 40)
for tool in ("obabel", "acpype", "namd3", "vmd"):
    print(f"{tool:<12} {shutil.which(tool) or 'not on PATH'}")

## 0.3 Install whatever is missing

This cell installs into the **current kernel's environment** (`sys.executable -m pip`),
which works both on Binder (where the kernel is the conda env repo2docker built) and in
any local Jupyter. It is safe to re-run: already-satisfied requirements are skipped.

On Binder the heavy conda packages (`rdkit`, `openbabel`) are expected to come from
`notebooks/environment.yml` at image build time — see `notebooks/README.md`. The pip
fallback below covers local use and lighter images.

In [ ]:
import importlib, shutil, subprocess, sys

PIP = [sys.executable, "-m", "pip", "install", "--quiet"]

for mod, dist in (("numpy", "numpy"),
                  ("matplotlib", "matplotlib"),
                  ("nglview", "nglview"),
                  ("rdkit", "rdkit"),          # wheels exist for py>=3.9; else use conda
                  ):
    try:
        importlib.import_module(mod)
        print(f"{mod:<12} already installed")
    except ImportError:
        print(f"{mod:<12} installing {dist} ...")
        r = subprocess.run(PIP + [dist], capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  pip could not install {dist}; on conda use:\n"
                  f"    conda install -c conda-forge {mod}")

for tool, dist in (("acpype", "acpype"),):
    if shutil.which(tool):
        print(f"{tool:<12} already installed")
    else:
        print(f"{tool:<12} installing {dist} ...")
        subprocess.run(PIP + [dist], capture_output=True, text=True)

# openbabel's CLI; pip's 'openbabel-wheel' ships the python bindings only on some
# platforms, so prefer conda for the CLI.  Notebook 02 refuses gracefully without it.
if not shutil.which("obabel"):
    print("obabel     not found -- needed for ligand extraction in notebook 02.\n"
          "           conda install -c conda-forge openbabel   (it is in the Binder env)")

## 0.4 Enable the nglview widget and smoke-test a 3D view

nglview needs its notebook extension registered once per environment. On JupyterLab ≥ 3
the widget is a prebuilt extension; the command below is a harmless no-op there.

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, "-m", "nglview", "install"],
                   capture_output=True, text=True)
print(r.stdout.strip() or "(no output)")
r = subprocess.run([sys.executable, "-m", "nglview", "enable"],
                   capture_output=True, text=True)
print(r.stdout.strip() or "(no output)")

import nglview as nv

# A trivial smoke test: a small protein drawn as cartoon + hetero groups.
view = nv.show_pdbid("1crn")
view.clear_representations()
view.add_representation("cartoon", color="sstruc")
view.add_representation("licorice", selection="hetero and not water")
view.center()
print("nglview OK — you should see crambin below.")
view

## 0.5 Locate the repository and put `bin/` on PATH

All notebooks assume the repository root is the parent of this notebook's directory —
true on Binder and for any normal clone.

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().resolve()
if not (REPO / "bin" / "rbfe").exists():        # running from notebooks/
    REPO = REPO.parent
assert (REPO / "bin" / "rbfe").exists(), f"cannot find bin/rbfe above {Path.cwd()}"

os.environ["PATH"] = str(REPO / "bin") + os.pathsep + os.environ["PATH"]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))               # so `import rbfe` works in-process

print("repo root :", REPO)
print("rbfe      :", REPO / "bin" / "rbfe")

import subprocess
r = subprocess.run(["rbfe", "--help"], capture_output=True, text=True)
print()
print(r.stdout.splitlines()[0] if r.returncode == 0 else r.stderr)
r = subprocess.run(["rbfe-prep", "--help"], capture_output=True, text=True)
print(r.stdout.splitlines()[0] if r.returncode == 0 else r.stderr)

## 0.6 A shared scratch directory for the tutorial

The notebooks write their tutorial system under `_tutorial/` in the repository root.
It is git-ignored, and re-running a notebook re-uses what is already there.

In [ ]:
from pathlib import Path

WORK = REPO / "_tutorial"
(WORK / "inputs").mkdir(parents=True, exist_ok=True)
print("tutorial workspace:", WORK)
print()
print("Next: open 01_complex_visualisation.ipynb")